In [ ]:
# import os
# import pydicom
# from dicompylercore import dicomparser

# # --- CONFIGURATION ---
# base_path = '../Data/SQUARE/'

# # Get the first patient folder automatically
# if os.path.exists(base_path):
#     patients = sorted([f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))])
#     if patients:
#         test_patient_id = patients[0] # Pick the first one
#         patient_dir = os.path.join(base_path, test_patient_id)
#         print(f"--- AUDITING SQUARE PATIENT: {test_patient_id} ---")
        
#         # 1. Find RTSTRUCT File
#         rtstruct_path = None
#         for root, dirs, files in os.walk(patient_dir):
#             for f in files:
#                 full_path = os.path.join(root, f)
#                 try:
#                     dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
#                     if dcm.get("Modality") == 'RTSTRUCT':
#                         rtstruct_path = full_path
#                         print(f"Found RTSTRUCT: {f}")
#                         break
#                 except:
#                     continue
#             if rtstruct_path: break
            
#         # 2. List Structures
#         if rtstruct_path:
#             try:
#                 rtss = dicomparser.DicomParser(rtstruct_path)
#                 structures = rtss.GetStructures()
#                 print(f"\n[STRUCTURE LIST]")
#                 for key, struct in structures.items():
#                     print(f"  ID {key}: {struct['name']}")
                    
#                 # Check for Lungs specifically
#                 lung_keywords = ['lung', 'total', 'combined', 'both']
#                 found_lungs = [s['name'] for k,s in structures.items() if any(x in s['name'].lower() for x in lung_keywords)]
                
#                 print(f"\n[ANALYSIS]")
#                 if found_lungs:
#                     print(f"✅ Found potential Lung structures: {found_lungs}")
#                 else:
#                     print(f"❌ No obvious 'Lung' structures found.")
                    
#             except Exception as e:
#                 print(f"Error parsing structures: {e}")
#         else:
#             print("❌ No RTSTRUCT file found in this patient folder.")
#     else:
#         print("No patient folders found in Square directory.")
# else:
#     print(f"Square Data Path not found: {base_path}")

In [ ]:
# import os
# import pydicom
# from dicompylercore import dicomparser

# # --- CONFIGURATION ---
# base_path = '../Data/SQUARE/'

# print(f"--- FAST AUDIT: SQUARE DATASET ---")

# if os.path.exists(base_path):
#     # Get all patient folders
#     patients = sorted([f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))])
    
#     if patients:
#         # Check ONLY the FIRST patient to start
#         test_patient = patients[0]
#         print(f"\nScanning Patient 1/1: {test_patient}")
#         patient_dir = os.path.join(base_path, test_patient)
        
#         rtstruct_path = None
#         file_count = 0
        
#         # 1. Find RTSTRUCT File
#         for root, dirs, files in os.walk(patient_dir):
#             for f in files:
#                 file_count += 1
#                 # print progress every 10 files so you know it's working
#                 if file_count % 10 == 0:
#                     print(f"  Scanned {file_count} files...", end='\r')
                    
#                 full_path = os.path.join(root, f)
#                 try:
#                     # force=True to handle missing headers
#                     dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
#                     if dcm.get("Modality") == 'RTSTRUCT':
#                         rtstruct_path = full_path
#                         print(f"\n  [FOUND] RTSTRUCT: {f}")
#                         break # STOP searching once found
#                 except:
#                     continue
#             if rtstruct_path: break
            
#         # 2. List Structures
#         if rtstruct_path:
#             try:
#                 rtss = dicomparser.DicomParser(rtstruct_path)
#                 structures = rtss.GetStructures()
                
#                 print(f"\n  [STRUCTURE LIST]")
#                 found_lung = False
#                 for key, struct in structures.items():
#                     name = struct['name']
#                     # Print names to check for Total Lung
#                     print(f"    - {name}")
                    
#                     if 'total' in name.lower() and 'lung' in name.lower():
#                         found_lung = True
#                     if 'both' in name.lower() and 'lung' in name.lower():
#                         found_lung = True
                        
#                 if found_lung:
#                     print("\n  [VERDICT] ✅ Found 'Total Lung' or 'Both Lungs'!")
#                 else:
#                     print("\n  [VERDICT] ⚠️ Only Separate Lungs (Lt/Rt) or Tumor found.")
                    
#             except Exception as e:
#                 print(f"  Error parsing structures: {e}")
#         else:
#             print(f"\n  ❌ No RTSTRUCT file found after scanning {file_count} files.")
#     else:
#         print("No patient folders found in Square directory.")
# else:
#     print(f"Square Data Path not found: {base_path}")

--- FAST AUDIT: SQUARE DATASET ---

Scanning Patient 1/1: R1508007367
  Scanned 170 files...
  [FOUND] RTSTRUCT: RS.R1508007367.CT_1.dcm

  [STRUCTURE LIST]
    - BODY
    - HEART
    - GTV
    - CTV NODE
    - CTV
    - CORD
    - BONES
    - LAD
    - LUNG_LT
    - LUNG_RT
    - PTV
    - TRACHEA
    - PTV node
    - CTV Plan
    - PTV Plan
    - NS_Ring
    - S
    - X
    - NS_Ring60

  [VERDICT] ⚠️ Only Separate Lungs (Lt/Rt) or Tumor found.


In [1]:
import os
import pandas as pd
import pydicom
from dicompylercore import dicomparser, dvhcalc
import warnings

warnings.filterwarnings("ignore")

# Define Square Path
data_sources = {
    'Square': '../Data/SQUARE_F/' 
}

output_csv = '../Results/square_dosiomics.csv'
os.makedirs('../Results', exist_ok=True)

print("Configured for SQUARE Target Extraction.")

Configured for SQUARE Target Extraction.


In [2]:
def find_target_roi_square(structure_dict):
    """
    Prioritizes PTV, then GTV.
    Returns: (roi_id, roi_name, roi_type)
    """
    # Priority 1: PTV (Planning Target Volume) - Best for D95 coverage
    ptv_names = ['ptv', 'ptv_total', 'ptv total', 'ptv_plan', 'ptv plan', 'ptv_primary']
    
    # Priority 2: GTV (Gross Tumor Volume)
    gtv_names = ['gtv', 'gtv_t', 'gtv t', 'gtv_total']

    # Clean available names
    available_rois = {}
    for key, struct in structure_dict.items():
        clean = struct['name'].lower().strip().replace(' ', '').replace('_', '').replace('-', '')
        available_rois[key] = clean

    # Check PTV First
    for name in ptv_names:
        clean_target = name.replace(' ', '').replace('_', '').replace('-', '')
        for key, clean_avail in available_rois.items():
            # Exact match or substring match (e.g. 'PTV_4500' matches 'PTV')
            if clean_target in clean_avail:
                return key, structure_dict[key]['name'], 'Target_PTV'

    # Check GTV Second
    for name in gtv_names:
        clean_target = name.replace(' ', '').replace('_', '').replace('-', '')
        for key, clean_avail in available_rois.items():
            if clean_target in clean_avail: 
                return key, structure_dict[key]['name'], 'Target_GTV'

    return None, None, None

In [3]:
results_list = []

for source_name, source_path in data_sources.items():
    if not os.path.exists(source_path):
        print(f"[ERROR] Folder not found: {source_path}")
        continue
        
    patient_folders = sorted([f for f in os.listdir(source_path) if os.path.isdir(os.path.join(source_path, f))])
    print(f"Processing {len(patient_folders)} patients in {source_name}...")
    
    for i, patient_id in enumerate(patient_folders):
        patient_dir = os.path.join(source_path, patient_id)
        
        rtstruct_path = None
        rtdose_path = None
        
        # Deep Search
        for root, dirs, files in os.walk(patient_dir):
            for f in files:
                full_path = os.path.join(root, f)
                try:
                    dcm = pydicom.dcmread(full_path, stop_before_pixels=True, force=True)
                    mod = dcm.get("Modality", "Unknown")
                    if mod == 'RTSTRUCT':
                        rtstruct_path = full_path
                    elif mod == 'RTDOSE':
                        if rtdose_path is None: rtdose_path = full_path
                        elif 'sum' in f.lower() or 'total' in f.lower(): rtdose_path = full_path
                except:
                    continue
            if rtstruct_path and rtdose_path: break 
        
        # Extract
        if rtstruct_path and rtdose_path:
            try:
                rtss = dicomparser.DicomParser(rtstruct_path)
                structures = rtss.GetStructures()
                
                # Find PTV/GTV
                roi_id, roi_name, roi_type = find_target_roi_square(structures)
                
                if roi_id:
                    dvh = dvhcalc.get_dvh(rtstruct_path, rtdose_path, roi_id)
                    if dvh:
                        metrics = {
                            'PatientID': patient_id,
                            'Source': source_name,
                            'ROI_Name': roi_name,
                            'ROI_Type': roi_type,
                            'Mean_Dose_Gy': dvh.mean,
                            'Max_Dose_Gy': dvh.max,
                            'Min_Dose_Gy': dvh.min,
                            'D95_Gy': dvh.dose_constraint(95).value # Critical for PTV
                        }
                        results_list.append(metrics)
                        print(f"[{i+1}] {patient_id}: Success ({roi_name})")
                    else:
                        print(f"[{i+1}] {patient_id}: [FAIL] Empty DVH")
                else:
                    avail = [s['name'] for k,s in structures.items()]
                    print(f"[{i+1}] {patient_id}: [SKIP] No Target Found. Avail: {avail[:3]}")
            except Exception as e:
                print(f"[{i+1}] {patient_id}: [ERROR] {str(e)}")
        else:
            print(f"[{i+1}] {patient_id}: [SKIP] Missing Files")

Processing 121 patients in Square...
[1] R130505087: Success (PTV)
[2] R1406007291: Success (PTV)
[3] R1508007367: Success (PTV)
[4] R1701004331: Success (PTV)
[5] R1702006414: Success (PTV)
[6] R1707004823: Success (PTV)
[7] R1710009613: Success (PTV)
[8] R1803003706: Success (PTV)
[9] R1807000334: Success (PTV)
[10] R1807003460: Success (PTV 60)
[11] R1807005275: Success (PTV)
[12] R1807007064: Success (PTV_Lung)
[13] R1808004062: Success (PTV60)
[14] R1810004752: Success (PTV)
[15] R1810006592: Success (PTV)
[16] R1810007735: Success (PTV)
[17] R1811001641: Success (PTV)
[18] R1903001183: Success (PTV)
[19] R1903004587: Success (PTV)
[20] R1905000571: Success (PTV)
[21] R1905002341: Success (PTV)
[22] R1905004532: Success (PTV lung)
[23] R1907005733: Success (PTV)
[24] R1907009307: Success (PTV)
[25] R1908007052: Success (PTV)
[26] R1908007690: Success (PTV)
[27] R1909004391: Success (PTV)
[28] R1909005566: Success (PTV)
[29] R1910000632: Success (PTV)
[30] R1910001778: Success (PTV

In [4]:
if results_list:
    df = pd.DataFrame(results_list)
    df.to_csv(output_csv, index=False)
    print(f"\n--- SUCCESS ---")
    print(f"Saved {len(df)} patients to {output_csv}")
    print(df.head())
else:
    print("\nNo data extracted.")


--- SUCCESS ---
Saved 121 patients to ../Results/square_dosiomics.csv
     PatientID  Source ROI_Name    ROI_Type  Mean_Dose_Gy  Max_Dose_Gy  \
0   R130505087  Square      PTV  Target_PTV     58.894961        64.42   
1  R1406007291  Square      PTV  Target_PTV     60.143217        61.65   
2  R1508007367  Square      PTV  Target_PTV     59.825628        61.61   
3  R1701004331  Square      PTV  Target_PTV     61.384563        63.38   
4  R1702006414  Square      PTV  Target_PTV     30.596897        32.03   

   Min_Dose_Gy  D95_Gy  
0        42.56   53.72  
1        53.58   58.98  
2        52.79   58.18  
3        51.16   58.94  
4        19.93   29.81  


In [5]:
import os
import pydicom
from dicompylercore import dicomparser

# --- CONFIGURATION ---
base_path = '../Data/SQUARE/'

print("--- INSPECTING SKIPPED PATIENTS ---")

skipped_count = 0

if os.path.exists(base_path):
    patients = sorted([f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f))])
    
    for patient_id in patients:
        # Stop after inspecting 5 skipped patients
        if skipped_count >= 5: break
        
        patient_dir = os.path.join(base_path, patient_id)
        
        # Find RTSTRUCT
        rtstruct_path = None
        for root, dirs, files in os.walk(patient_dir):
            for f in files:
                try:
                    dcm = pydicom.dcmread(os.path.join(root, f), stop_before_pixels=True, force=True)
                    if dcm.get("Modality") == 'RTSTRUCT':
                        rtstruct_path = os.path.join(root, f)
                        break
                except: continue
            if rtstruct_path: break
            
        if rtstruct_path:
            try:
                rtss = dicomparser.DicomParser(rtstruct_path)
                structures = rtss.GetStructures()
                
                # Check if our previous code would have found it
                # (We copy the previous logic here to test)
                ptv_names = ['ptv', 'ptv_total', 'ptv total', 'ptv_plan', 'ptv plan', 'ptv_primary']
                gtv_names = ['gtv', 'gtv_t', 'gtv t', 'gtv_total']
                
                found = False
                avail_clean = [s['name'].lower().strip().replace(' ','').replace('_','').replace('-','') for k,s in structures.items()]
                
                # Test PTV
                for n in ptv_names:
                    clean_n = n.replace(' ','').replace('_','').replace('-','')
                    if any(clean_n in a for a in avail_clean): found = True
                
                # Test GTV
                if not found:
                    for n in gtv_names:
                        clean_n = n.replace(' ','').replace('_','').replace('-','')
                        if any(clean_n in a for a in avail_clean): found = True
                
                # If NOT found, print it so we can add it!
                if not found:
                    print(f"\nSkipped Patient: {patient_id}")
                    print(f"  Available Structures: {[s['name'] for k,s in structures.items()]}")
                    skipped_count += 1
                    
            except: continue

--- INSPECTING SKIPPED PATIENTS ---
